# Сложные операции над объектами Pandas

На прошлом занятии мы изучили простые операции над объектами Pandas: векторизованные операции, операции для очистки данных, а также операции, которые могут быть полезны для преобразования данных в отдельных столбцах. В этом занятии мы рассмотрим более комплексные операции над объектами Pandas. Эти операции включают в себя объединение датафреймов по индексам и колонкам, а также агрегирование данных.

**Важно**:  
Перед началом работы обязательно установите зависимости из файла [requirements.txt](./requirements.txt).

**Необходимые импорты**:

In [1]:
from collections.abc import Sequence
from typing import (
    Any,
    Final,
)
from uuid import uuid4

import numpy as np
import pandas as pd
from faker import Faker

**Вспомогательные объекты**:

In [2]:
faker: Final[Faker] = Faker()

In [3]:
def create_df_from_index_and_columns(
    index: Sequence[Any],
    columns: Sequence[Any],
) -> pd.DataFrame:
    data = {
        column_name: [
            f"{index_name}|{column_name}" for index_name in index
        ]
        for column_name in columns
    }

    return pd.DataFrame(data, index=index)

## Объединение данных

Очень часто при решении практических задач вам придется иметь дело не с одним единственным источником данных, а с несколькими. Данные могут храниться в разных csv-файлах, таблицах в БД и даже на разных серверах. Поэтому важно уметь правильно объединять данные из разных источников в единый датафрейм для удобства дальнейшей работы (конечно, если это целесообразно в рамках решаемой задачи). Итак, в этой секции рассмотрим наиболее распространенные способы объединения данных в Pandas.

### Конкатенация

Конкатенация - это простейшая форма объединения двух и более наборов данных в единый набор. Несмотря на свою простоту, конкатенация бывает очень полезна в тех случаях, когда вам нужно объединить данные с одинаковой структурой, полученные из разных источников. Примером таких данных может быть некоторая таблица, которая хранится в виде набора csv-файлов с одинаковой структурой. Чтобы восстановить исходную таблицу из данного набора csv-файлов, нам достаточно просто прочитать содержимое этих файлов и выполнить конкатенацию полученных датафреймов.

Конкатенация в Pandas напоминает конкатенацию в NumPy и выполняется с помощью функции `pd.concat`. Обязательный аргумент функции `pd.concat` - итерируемый объект `objs` с объектами, которые требуется объединить. С помощью этой функции вы можете объединить два и более набора данных (`pd.Series` или `pd.DataFrame`), записав данные из них последовательно, друг за другом.

Простейший вариант использования функции `pd.concat` - конкатенация двух объектов `pd.Series`:

In [4]:
series_size = 3

series1 = pd.Series(
    np.random.randint(170, 190, size=series_size, dtype=np.uint8),
    index=[str(uuid4()) for _ in range(series_size)]
)
series2 = pd.Series(
    np.random.randint(150, 180, size=series_size, dtype=np.uint8),
    index=[str(uuid4()) for _ in range(series_size)]
)
series_concatenated = pd.concat((series1, series2))

print(
    f"series1:\n{series1}",
    f"series2:\n{series2}",
    f"concatenation result:\n{series_concatenated}",
    sep="\n\n",
)

series1:
76cad633-264a-4843-93e9-5e106aa9459e    175
0a88002b-792d-45b4-b7c5-ba0e9fa7e262    175
02f8bda2-ddb3-4c01-9faf-1e1d29704d2e    178
dtype: uint8

series2:
899f37c1-e686-441e-85fe-3fc906b850af    173
e5900fd4-829b-4b15-8a78-b3ea7fe4d390    160
17139b8a-8f2b-42a7-bcc1-021546e14fc1    171
dtype: uint8

concatenation result:
76cad633-264a-4843-93e9-5e106aa9459e    175
0a88002b-792d-45b4-b7c5-ba0e9fa7e262    175
02f8bda2-ddb3-4c01-9faf-1e1d29704d2e    178
899f37c1-e686-441e-85fe-3fc906b850af    173
e5900fd4-829b-4b15-8a78-b3ea7fe4d390    160
17139b8a-8f2b-42a7-bcc1-021546e14fc1    171
dtype: uint8


Функция `pd.concat` также может быть использована для объединения более сложных объектов, типа `pd.DataFrame`:

In [5]:
columns = ["col1", "col2"]

dataframe1 = create_df_from_index_and_columns(
    columns=columns,
    index=[1, 2],
)
dataframe2 = create_df_from_index_and_columns(
    columns=columns,
    index=[3, 4],
)
dataframe_concatenated = pd.concat((dataframe1, dataframe2))

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

dataframe1:
     col1    col2
1  1|col1  1|col2
2  2|col1  2|col2

dataframe2:
     col1    col2
3  3|col1  3|col2
4  4|col1  4|col2

concatenation result:
     col1    col2
1  1|col1  1|col2
2  2|col1  2|col2
3  3|col1  3|col2
4  4|col1  4|col2


По умолчанию объединение объектов `pd.Dataframe` с помощью `pd.concat` выполняется построчно. Однако, при объединении данных высокой размерности `pd.concat` предоставляет возможность настройки, как именно будут объединены данные. За эту настройку отвечает знакомый нам аргумент `axis`. Этот аргумент может быть связан как с целочисленными значениями, как мы уже видели при изучении NumPy, так и со строковыми литералами, типа `"columns"`. Использование строковых литералов предпочтительнее из-за увеличения понятности кода.

In [6]:
index = ["row1", "row2"]

dataframe1 = create_df_from_index_and_columns(
    columns=["col1", "col2"],
    index=index,
)
dataframe2 = create_df_from_index_and_columns(
    columns=["col3", "col4"],
    index=index,
)
dataframe_concatenated = pd.concat(
    (dataframe1, dataframe2),
    axis="columns",
)

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

dataframe1:
           col1       col2
row1  row1|col1  row1|col2
row2  row2|col1  row2|col2

dataframe2:
           col3       col4
row1  row1|col3  row1|col4
row2  row2|col3  row2|col4

concatenation result:
           col1       col2       col3       col4
row1  row1|col1  row1|col2  row1|col3  row1|col4
row2  row2|col1  row2|col2  row2|col3  row2|col4


Во всех предыдущих примерах мы предполагали, что объединяемые объекты имеют или одни и те же индексы, или одни и те же имена столбцов. Однако `pd.concat` будет корректно работать и в тех случаях, когда это не так. Если объединяемые данные имеют разный набор индексов и столбцов, место недостающих данных будет заполнено `NaN` значениями.

In [7]:
dataframe1 = create_df_from_index_and_columns(
    columns=["col1", "col2"],
    index=["row1", "row2"],
)
dataframe2 = create_df_from_index_and_columns(
    columns=["col3", "col4"],
    index=["row3", "row4"],
)
dataframe_concatenated = pd.concat((dataframe1, dataframe2))

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

dataframe1:
           col1       col2
row1  row1|col1  row1|col2
row2  row2|col1  row2|col2

dataframe2:
           col3       col4
row3  row3|col3  row3|col4
row4  row4|col3  row4|col4

concatenation result:
           col1       col2       col3       col4
row1  row1|col1  row1|col2        NaN        NaN
row2  row2|col1  row2|col2        NaN        NaN
row3        NaN        NaN  row3|col3  row3|col4
row4        NaN        NaN  row4|col3  row4|col4


#### Борьба с дублированием индексов

После объединения набора данных функция `pd.concat` сохраняет исходные индексы. Данное поведение может приводить к появлению дублирующихся индексов.

В приведенном примере результирующий `pd.DataFrame` будет содержать два значения `"row2"` в своем индексе.

In [8]:
columns = ["col1", "col2"]

dataframe1 = create_df_from_index_and_columns(
    columns=columns,
    index=["row1", "row2"],
)
dataframe2 = create_df_from_index_and_columns(
    columns=columns,
    index=["row2", "row3"],
)
dataframe_concatenated = pd.concat((dataframe1, dataframe2))

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

dataframe1:
           col1       col2
row1  row1|col1  row1|col2
row2  row2|col1  row2|col2

dataframe2:
           col1       col2
row2  row2|col1  row2|col2
row3  row3|col1  row3|col2

concatenation result:
           col1       col2
row1  row1|col1  row1|col2
row2  row2|col1  row2|col2
row2  row2|col1  row2|col2
row3  row3|col1  row3|col2


В некоторых ситуациях такое дублирование нежелательно. `pd.concat` предоставляет два способа по борьбе с дублирующимися индексами: игнорирование индексов исходных наборов данных и проверка уникальности.

При игнорировании индексов исходных наборов данных результат конкатенации будет содержать новый индекс, сгенерированный автоматически. Чтобы воспользоваться данным способом, необходимо вызвать функцию `pd.concat` с флагом `ignore_index=True`.

In [9]:
columns = ["col1", "col2"]

dataframe1 = create_df_from_index_and_columns(
    columns=columns,
    index=["row1", "row2"],
)
dataframe2 = create_df_from_index_and_columns(
    columns=columns,
    index=["row2", "row3"],
)
dataframe_concatenated = pd.concat(
    (dataframe1, dataframe2),
    ignore_index=True,
)

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

dataframe1:
           col1       col2
row1  row1|col1  row1|col2
row2  row2|col1  row2|col2

dataframe2:
           col1       col2
row2  row2|col1  row2|col2
row3  row3|col1  row3|col2

concatenation result:
        col1       col2
0  row1|col1  row1|col2
1  row2|col1  row2|col2
2  row2|col1  row2|col2
3  row3|col1  row3|col2


В этом случае индекс не будет содержать никаких дубликатов, однако при этом вы теряете значения исходных индексов. Поэтому данный способ борьбы с дубликатами подходит в том случае, если значения индекса неважны для решения задачи.

Второй способ борьбы с дубликатами - проверка уникальности. Чтобы воспользоваться данным способом, необходимо вызвать функцию `pd.concat` с флагом `verify_integrity=True`.

In [10]:
columns = ["col1", "col2"]

dataframe1 = create_df_from_index_and_columns(
    columns=columns,
    index=["row1", "row2"],
)
dataframe2 = create_df_from_index_and_columns(
    columns=columns,
    index=["row2", "row3"],
)
# этот вызов должен привести к ошибке ValueError
dataframe_concatenated = pd.concat(
    (dataframe1, dataframe2),
    verify_integrity=True,
)

ValueError: Indexes have overlapping values: Index(['row2'], dtype='object')

При использовании флага `verify_integrity=True` вызов функции `pd.concat` завершится успехом в том и только в том случае, если индексы исходных наборов данных не имели пересечения. В противном случае вы получите ошибку `ValueError`.

При использовании данного подхода в результате конкатенации сохраняются значения индексов исходных наборов данных.

Существует еще один способ борьбы с дубликатами - создание мульти-индекса. Однако этот способ в нашем курсе рассматриваться не будет (как и мульти-индексы). Если вы заинтересовались темой мульти-индексов, вы можете ознакомиться с ней самостоятельно в [официальной документации](https://pandas.pydata.org/pandas-docs/version/2.2/reference/api/pandas.MultiIndex.html).

### Сложные объединения данных

Сложные объединения данных в Pandas являются реализациями правил реляционной алгебры по объединению данных. Звучит страшно и сложно, но на практике все не так пугающе, как кажется на первый взгляд. Те из вас, кто знаком с SQL, увидят в описанных функция операцию `join`.

Сложные объединения данных незаменимы в тех случаях, когда данные, полученные из разных источников (например, из разных файлов), имеют сложные системы связей. Например, у нас может быть датасет с данными о пользователях и отдельный датасет с данными об активности пользователей на нашем сайте. Сами по себе датасеты имеют разную структуру и связаны между собой через значения колонки `user_id`. В этом случае для комплексного анализа данных об активности пользователей нашего сайта нам потребуется сложное объединение этих данных, а не простая конкатенация.

В Pandas сложное объединение наборов данных может быть выполнено с помощью функции `pd.merge`.

In [17]:
usernames = [faker.user_name() for _ in range(4)]
user_ids = [str(uuid4()) for _ in usernames]

user_data = pd.DataFrame(
    {
        "user_id": user_ids,
        "username": usernames,
    },
)

orders_amount = 8
order_data = pd.DataFrame(
    {
        "user_id": np.random.choice(user_ids, size=orders_amount).tolist(),
        "order_id": [str(uuid4()) for _ in range(orders_amount)],
    },
)

print(
    f"user_data:\n{user_data}",
    f"order_data:\n{order_data}",
    sep="\n\n",
)

user_data:
                                user_id         username
0  f246f70d-461b-4a49-99cd-55aa8ffe732e    christinarice
1  6a3fe529-a1cb-43be-b3f6-1f1a3afadcb4        kgarrison
2  cca35637-60e9-49f3-9501-66ff4250dd68  richardsrichard
3  40fa13f5-ea7e-4b9f-817c-aa8d31508320  thompsonrebecca

order_data:
                                user_id                              order_id
0  40fa13f5-ea7e-4b9f-817c-aa8d31508320  fcac972c-4164-4158-8a62-fce19037b0e7
1  40fa13f5-ea7e-4b9f-817c-aa8d31508320  c280bd1c-0dea-4fce-be15-c9d3c009f74b
2  f246f70d-461b-4a49-99cd-55aa8ffe732e  bac355fe-84b9-4d97-88d8-a3fe7aed6e04
3  6a3fe529-a1cb-43be-b3f6-1f1a3afadcb4  fa8430a9-3ac8-4311-bb7a-9dbf3d1d2528
4  40fa13f5-ea7e-4b9f-817c-aa8d31508320  1a617a85-9c91-43cd-bf64-d6096b00454c
5  40fa13f5-ea7e-4b9f-817c-aa8d31508320  755fd1be-72b1-41f8-bdf1-cc279c297ff1
6  6a3fe529-a1cb-43be-b3f6-1f1a3afadcb4  b6357224-e5c4-4bd2-a44c-fd14792ff7eb
7  6a3fe529-a1cb-43be-b3f6-1f1a3afadcb4  a8f035ad-f31b-4828-8601-e8

In [18]:
pd.merge(user_data, order_data)

,user_id,username,order_id
0,f246f70d-461b-4a49-99cd-55aa8ffe732e,christinarice,bac355fe-84b9-4d97-88d8-a3fe7aed6e04
1,6a3fe529-a1cb-43be-b3f6-1f1a3afadcb4,kgarrison,fa8430a9-3ac8-4311-bb7a-9dbf3d1d2528
2,6a3fe529-a1cb-43be-b3f6-1f1a3afadcb4,kgarrison,b6357224-e5c4-4bd2-a44c-fd14792ff7eb
3,6a3fe529-a1cb-43be-b3f6-1f1a3afadcb4,kgarrison,a8f035ad-f31b-4828-8601-e8cc8c3b81b4
4,40fa13f5-ea7e-4b9f-817c-aa8d31508320,thompsonrebecca,fcac972c-4164-4158-8a62-fce19037b0e7
5,40fa13f5-ea7e-4b9f-817c-aa8d31508320,thompsonrebecca,c280bd1c-0dea-4fce-be15-c9d3c009f74b
6,40fa13f5-ea7e-4b9f-817c-aa8d31508320,thompsonrebecca,1a617a85-9c91-43cd-bf64-d6096b00454c
7,40fa13f5-ea7e-4b9f-817c-aa8d31508320,thompsonrebecca,755fd1be-72b1-41f8-bdf1-cc279c297ff1
